# hp tuning of prototype model for mvp

In [1]:
import sys
import os
from dotenv import load_dotenv
# Add the project root directory to Python path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(project_root)

from scripts.utils import get_time_series, get_timestamp_index, cleaning_data, feature_engineering, model_errors, plot_prediction, calculate_errors

In [2]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.ensemble import RandomForestRegressor
import optuna
import mlflow
import mlflow.sklearn
import boto3
import logging
from datetime import datetime
random.seed(42)         # Python built-in random module
np.random.seed(42)

In [3]:
# 로깅 설정
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(f"regression_log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"),
        logging.StreamHandler()  # 콘솔에도 출력
    ]
)
logger = logging.getLogger(__name__)

# 데이터 로드
logger.info("데이터 MVP 단계 시작")

2025-04-10 11:20:01,124 - INFO - 데이터 MVP 단계 시작


In [4]:
load_dotenv()

True

In [5]:
# MLflow 설정
mlflow.set_tracking_uri("http://localhost:5001")  # MLflow Tracking Server URI (호스트 기준)
mlflow.set_experiment("Delay_Prediction_MVP_Experiment")  # 실험 이름 설정

2025/04/10 11:20:01 INFO mlflow.tracking.fluent: Experiment with name 'Delay_Prediction_MVP_Experiment' does not exist. Creating a new experiment.


<Experiment: artifact_location='s3://artifact-store-sun/mlruns/1', creation_time=1744276801253, experiment_id='1', last_update_time=1744276801253, lifecycle_stage='active', name='Delay_Prediction_MVP_Experiment', tags={}>

In [6]:
# 데이터 로드
logger.info("데이터 로드, cleaning, feature engineering 시작")
raw_data = pd.read_csv('../raw_data_20250403.csv')
data = get_timestamp_index(raw_data)
region = 'Seoul'
cleaned_region = cleaning_data(data, region)
cleaned_feature_added = feature_engineering(cleaned_region)
logger.info(f"{region} 데이터 로드 성공: {cleaned_feature_added.shape[0]} 행, {cleaned_feature_added.shape[1]} 열")

2025-04-10 11:20:01,316 - INFO - 데이터 로드, cleaning, feature engineering 시작
/Volumes/SUNSE/projects/building-ml-pipeline/scripts/utils.py:70: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.
  data_resampled[numeric_time_features] = data_resampled[numeric_time_features].interpolate(method='time')
/Volumes/SUNSE/projects/building-ml-pipeline/scripts/utils.py:71: FutureWarning: DataFrame.interpolate with method=bfill is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  data_resampled[categorical_date_features] = data_resampled[categorical_date_features].interpolate(method='bfill')
/Volumes/SUNSE/projects/building-ml-pipeline/scripts/utils.py:74: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method 

In [7]:
# target log transformation
cleaned_feature_added['delay_hours_log'] = np.log1p(cleaned_feature_added['delay_hours'])

In [8]:
# based on feature selection from prototyping stage
features=['PTY', 'REH', 'RN1', 'T1H', 'WSD',  'day',  'hour', 
        'sin_hour', 'cos_hour', 'is_weekend',
       'day_of_week_encoded','PTY_lag1', 'PTY_lag2', 'delay_hours_lag1',
       'delay_hours_lag2']

X  = cleaned_feature_added[features]
y = cleaned_feature_added['delay_hours_log']

In [9]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False, random_state=42)
logger.info(f"훈련 데이터: {X_train.shape[0]} 행, 테스트 데이터: {X_test.shape[0]} 행")

2025-04-10 11:20:01,388 - INFO - 훈련 데이터: 300 행, 테스트 데이터: 76 행


In [10]:
# Optuna로 Hyperparameter Tuning
def objective(trial):
    n_estimators = trial.suggest_int('n_estimators', 100, 500)
    max_depth = trial.suggest_int('max_depth', 5, 20)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 5)
    max_features = trial.suggest_categorical('max_features', ['sqrt', 'log2', 0.5])

    model = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        random_state=42
    )

    tscv = TimeSeriesSplit(n_splits=5)
    rmses = []
    for train_idx, test_idx in tscv.split(X_train):
        X_train_cv, X_test_cv = X_train.iloc[train_idx], X_train.iloc[test_idx]
        y_train_cv, y_test_cv = y_train.iloc[train_idx], y_train.iloc[test_idx]
        model.fit(X_train_cv, y_train_cv)
        y_pred_cv = model.predict(X_test_cv)
        y_pred_cv_original = np.expm1(y_pred_cv)
        y_test_cv_original = np.expm1(y_test_cv)
        mse = mean_squared_error(y_test_cv_original, y_pred_cv_original)
        rmse = np.sqrt(mse)
        
        rmses.append(rmse)
    return np.mean(rmses)

In [11]:
# MLflow 실험 시작
with mlflow.start_run() as run:
    # Optuna Study 생성 및 최적화
    logger.info("Optuna Hyperparameter Tuning 시작")
    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=32)
    logger.info("Optuna Hyperparameter Tuning 완료")

    # Best parameters and score
    best_params = study.best_params
    best_rmse = study.best_value
    logger.info(f"Best parameters: {best_params}")
    logger.info(f"Best TimeSeriesSplit RMSE: {best_rmse:.4f}")

    # MLflow에 파라미터와 메트릭 로깅
    mlflow.log_params(best_params)
    mlflow.log_metric("cv_rmse", best_rmse)

    # 최종 모델 학습 (최적 파라미터 사용)
    logger.info("Final Random Forest 모델 학습 시작")
    final_model = RandomForestRegressor(**best_params, random_state=42)
    final_model.fit(X_train, y_train)
    logger.info("Final Random Forest 모델 학습 완료")

    # 예측 및 평가
    y_pred_log = final_model.predict(X_test)
    y_pred_original = np.expm1(y_pred_log)
    y_test_original = np.expm1(y_test)
    mse = mean_squared_error(y_test_original, y_pred_original)
    rmse = np.sqrt(mse)
    logger.info(f"Final Random Forest with tuned parameters (Optuna) - RMSE: {rmse:.4f}")

    # MLflow에 테스트 RMSE 로깅
    mlflow.log_metric("test_rmse", rmse)

    # 모델과 스케일러를 MLflow에 저장
    mlflow.sklearn.log_model(final_model, "random_forest_model")
    logger.info(f"모델과 MLflow를 통해 S3에 저장됨: {mlflow.get_artifact_uri()}")



2025-04-10 11:20:01,534 - INFO - Optuna Hyperparameter Tuning 시작
[I 2025-04-10 11:20:01,535] A new study created in memory with name: no-name-308de6cb-170d-4449-87b5-5cfdac16ef98
[I 2025-04-10 11:20:02,627] Trial 0 finished with value: 0.09672032897124108 and parameters: {'n_estimators': 303, 'max_depth': 20, 'min_samples_split': 2, 'min_samples_leaf': 4, 'max_features': 0.5}. Best is trial 0 with value: 0.09672032897124108.
[I 2025-04-10 11:20:03,763] Trial 1 finished with value: 0.09819269093666694 and parameters: {'n_estimators': 398, 'max_depth': 10, 'min_samples_split': 4, 'min_samples_leaf': 5, 'max_features': 'log2'}. Best is trial 0 with value: 0.09672032897124108.
[I 2025-04-10 11:20:04,445] Trial 2 finished with value: 0.09470742533657009 and parameters: {'n_estimators': 274, 'max_depth': 5, 'min_samples_split': 4, 'min_samples_leaf': 2, 'max_features': 'sqrt'}. Best is trial 2 with value: 0.09470742533657009.
[I 2025-04-10 11:20:05,347] Trial 3 finished with value: 0.0949197

🏃 View run upbeat-hog-587 at: http://localhost:5001/#/experiments/1/runs/7ae633a1402d494284d3cbafef148e70
🧪 View experiment at: http://localhost:5001/#/experiments/1
